# 01 Train CarDD YOLO11 Segmentation

This notebook is the full training pipeline for Colab.

Dependency order:

1. Run **Mount Drive**.
2. Run **Install and Import**.
3. Run **Configure Paths**.
4. Run **Download CarDD to Drive** after dataset access has been approved.
5. Run **Prepare Dataset** only after the authorized CarDD zip or extracted data is in Drive.
6. Run **Train or Resume** only after `data_ready.json` exists.
7. Run **Evaluate and Export** only after `best.pt` or `last.pt` exists.

All large files are written to `/content/drive/MyDrive/CarDD_YOLO11` so Colab disconnects do not lose checkpoints.

## Mount Drive

Dependency: none. Run this first in every Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install and Import

Dependency: Drive is mounted. This installs the runtime packages used by all later cells.

In [ ]:
%pip install -q ultralytics gdown opencv-python matplotlib pandas pyyaml tqdm

In [ ]:
import json
import os
import random
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import gdown
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Configure Paths

Dependency: imports are ready. Change only the variables in this cell if needed.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
DATA_RAW_ROOT = DRIVE_ROOT / 'data_raw'
DATA_COCO_ROOT = DRIVE_ROOT / 'data_coco'
DATA_YOLO_ROOT = DRIVE_ROOT / 'data_yolo'
RUNS_ROOT = DRIVE_ROOT / 'runs'
REPORTS_ROOT = DRIVE_ROOT / 'reports'
EXPORTS_ROOT = DRIVE_ROOT / 'exports'
BACKUPS_ROOT = DRIVE_ROOT / 'backups'

BASE_MODEL = 'yolo11n-seg.pt'
RUN_NAME = 'yolo11n_seg'
TRAIN_PROJECT = RUNS_ROOT / 'train'
VAL_PROJECT = RUNS_ROOT / 'val'

EPOCHS = 100
IMGSZ = 1024
BATCH = 16
WORKERS = 2
SEED = 42
FORCE_REBUILD_DATASET = False
FORCE_REDOWNLOAD_CARDD = False

CARDD_FILE_ID = '1bbyqVCKZX5Ur5Zg-uKj0jD0maWAVeOLx'
CARDD_URL = f'https://drive.google.com/uc?id={CARDD_FILE_ID}'
CARDD_ZIP = DATA_RAW_ROOT / 'CarDD_release.zip'

for path in [DATA_RAW_ROOT, DATA_COCO_ROOT, DATA_YOLO_ROOT, RUNS_ROOT, REPORTS_ROOT, EXPORTS_ROOT, BACKUPS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('YOLO dataset root:', DATA_YOLO_ROOT)
print('CarDD zip target:', CARDD_ZIP)

## Download CarDD to Drive

Dependency: Drive is mounted and paths are configured. Run this after the CarDD license request has been approved.

This cell is resumable: if `data_raw/CarDD_release.zip` already exists and `FORCE_REDOWNLOAD_CARDD=False`, it skips the large download.

In [ ]:
if CARDD_ZIP.exists() and CARDD_ZIP.stat().st_size > 0 and not FORCE_REDOWNLOAD_CARDD:
    print('CarDD zip already exists in Drive:', CARDD_ZIP)
    print('Size GB:', round(CARDD_ZIP.stat().st_size / 1024**3, 2))
else:
    if CARDD_ZIP.exists() and FORCE_REDOWNLOAD_CARDD:
        CARDD_ZIP.unlink()
    print('Downloading CarDD to Drive:', CARDD_ZIP)
    downloaded = gdown.download(CARDD_URL, str(CARDD_ZIP), quiet=False, fuzzy=False)
    if downloaded is None or not CARDD_ZIP.exists() or CARDD_ZIP.stat().st_size == 0:
        raise RuntimeError('CarDD download failed. Open the official link once in the browser, confirm access, then rerun this cell.')
    print('Downloaded:', CARDD_ZIP)
    print('Size GB:', round(CARDD_ZIP.stat().st_size / 1024**3, 2))

## Prepare Dataset

Dependency: authorized CarDD data is already in `data_raw` as a zip file, or extracted under `data_coco`.

This cell is resumable: if `data_yolo/data_ready.json` already exists and `FORCE_REBUILD_DATASET=False`, it skips conversion.

In [ ]:
DATA_READY = DATA_YOLO_ROOT / 'data_ready.json'
DATA_YAML = DATA_YOLO_ROOT / 'cardd.yaml'

def extract_first_zip_if_needed():
    if any(DATA_COCO_ROOT.rglob('*.json')):
        return
    zips = sorted(DATA_RAW_ROOT.glob('*.zip'))
    if not zips:
        return
    print('Extracting:', zips[0])
    with zipfile.ZipFile(zips[0], 'r') as zf:
        zf.extractall(DATA_COCO_ROOT)

def infer_split(path):
    name = path.name.lower()
    if 'train' in name:
        return 'train'
    if 'val' in name or 'valid' in name:
        return 'val'
    if 'test' in name:
        return 'test'
    return None

def find_annotation_files():
    files = []
    for root in [DATA_COCO_ROOT, DATA_RAW_ROOT]:
        files.extend(root.rglob('*.json'))
    split_to_file = {}
    for file in sorted(files):
        split = infer_split(file)
        if split and split not in split_to_file:
            split_to_file[split] = file
    if 'train' not in split_to_file:
        raise FileNotFoundError('No train annotation JSON found. Put the authorized CarDD COCO data under data_raw or data_coco.')
    if 'val' not in split_to_file and 'test' not in split_to_file:
        raise FileNotFoundError('Need at least a val or test annotation JSON for evaluation.')
    return split_to_file

def build_image_index():
    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    index = {}
    for root in [DATA_COCO_ROOT, DATA_RAW_ROOT]:
        for file in root.rglob('*'):
            if file.suffix.lower() in exts:
                index.setdefault(file.name, file)
                rel = str(file.relative_to(root)).replace('\\', '/')
                index.setdefault(rel, file)
    return index

def normalize_polygon(poly, width, height):
    values = []
    for i, value in enumerate(poly):
        limit = width if i % 2 == 0 else height
        norm = max(0.0, min(1.0, float(value) / float(limit)))
        values.append(norm)
    return values

def convert_one_split(split, ann_file, image_index, class_names=None):
    coco = json.loads(ann_file.read_text(encoding='utf-8'))
    categories = sorted(coco['categories'], key=lambda x: x['id'])
    if class_names is None:
        class_names = [c['name'] for c in categories]
    cat_to_cls = {cat['id']: idx for idx, cat in enumerate(categories)}
    images = {img['id']: img for img in coco['images']}
    anns_by_image = defaultdict(list)
    for ann in coco.get('annotations', []):
        anns_by_image[ann['image_id']].append(ann)

    out_img_dir = DATA_YOLO_ROOT / 'images' / split
    out_lbl_dir = DATA_YOLO_ROOT / 'labels' / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    labels = 0
    missing = []
    for image_id, image in tqdm(images.items(), desc=f'convert {split}'):
        file_name = image['file_name']
        src = image_index.get(file_name) or image_index.get(Path(file_name).name)
        if src is None:
            missing.append(file_name)
            continue

        dst_img = out_img_dir / Path(file_name).name
        if not dst_img.exists():
            shutil.copy2(src, dst_img)
            copied += 1

        width = image.get('width')
        height = image.get('height')
        if not width or not height:
            img = cv2.imread(str(dst_img))
            height, width = img.shape[:2]

        lines = []
        for ann in anns_by_image.get(image_id, []):
            seg = ann.get('segmentation')
            if not isinstance(seg, list) or not seg:
                continue
            poly = max(seg, key=len)
            if len(poly) < 6:
                continue
            cls = cat_to_cls[ann['category_id']]
            values = normalize_polygon(poly, width, height)
            lines.append(str(cls) + ' ' + ' '.join(f'{v:.6f}' for v in values))

        label_file = out_lbl_dir / (Path(file_name).stem + '.txt')
        label_file.write_text('\n'.join(lines), encoding='utf-8')
        labels += len(lines)

    if missing:
        print(f'Warning: {len(missing)} images missing in split {split}. First missing:', missing[:3])
    return {'images': len(images), 'copied': copied, 'labels': labels, 'annotation_file': str(ann_file)}, class_names

extract_first_zip_if_needed()

if DATA_READY.exists() and DATA_YAML.exists() and not FORCE_REBUILD_DATASET:
    print('Dataset already prepared:', DATA_READY)
    print(DATA_READY.read_text(encoding='utf-8'))
else:
    split_files = find_annotation_files()
    print('Annotation files:', split_files)
    image_index = build_image_index()
    print('Indexed images:', len(image_index))

    summary = {}
    class_names = None
    for split, ann_file in split_files.items():
        summary[split], class_names = convert_one_split(split, ann_file, image_index, class_names)

    val_split = 'val' if 'val' in summary else 'test'
    yaml_data = {
        'path': str(DATA_YOLO_ROOT),
        'train': 'images/train',
        'val': f'images/{val_split}',
        'names': {i: name for i, name in enumerate(class_names)},
    }
    if 'test' in summary:
        yaml_data['test'] = 'images/test'
    DATA_YAML.write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')
    DATA_READY.write_text(json.dumps({'summary': summary, 'names': class_names}, indent=2), encoding='utf-8')
    print(DATA_YAML.read_text(encoding='utf-8'))
    print('Prepared:', DATA_READY)

## Quick Label Visualization

Dependency: `data_ready.json` exists. This checks whether converted masks roughly align with images.

In [ ]:
assert DATA_READY.exists(), 'Run Prepare Dataset first.'
sample_images = list((DATA_YOLO_ROOT / 'images' / 'train').glob('*'))
sample = random.choice(sample_images)
label = DATA_YOLO_ROOT / 'labels' / 'train' / (sample.stem + '.txt')
img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
overlay = img.copy()
if label.exists():
    for line in label.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        pts = [float(x) for x in parts[1:]]
        xy = [(int(pts[i] * w), int(pts[i + 1] * h)) for i in range(0, len(pts), 2)]
        import numpy as np
        cv2.polylines(overlay, [np.array(xy, dtype=np.int32)], True, (255, 0, 0), 2)
plt.figure(figsize=(10, 8))
plt.imshow(overlay)
plt.axis('off')
plt.title(sample.name)
plt.show()

## Train or Resume

Dependency: `data_yolo/cardd.yaml` and `data_ready.json` exist.

Resume behavior: if `runs/train/yolo11n_seg/weights/last.pt` exists in Drive, this cell resumes automatically.

In [ ]:
assert DATA_READY.exists() and DATA_YAML.exists(), 'Run Prepare Dataset first.'
last_pt = TRAIN_PROJECT / RUN_NAME / 'weights' / 'last.pt'

if last_pt.exists():
    print('Resuming from:', last_pt)
    model = YOLO(str(last_pt))
    train_results = model.train(resume=True)
else:
    print('Starting new training from:', BASE_MODEL)
    model = YOLO(BASE_MODEL)
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        optimizer='AdamW',
        amp=True,
        overlap_mask=True,
        multi_scale=True,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
        patience=30,
        save_period=1,
        workers=WORKERS,
        seed=SEED,
        project=str(TRAIN_PROJECT),
        name=RUN_NAME,
        exist_ok=True,
    )

print('Training output:', TRAIN_PROJECT / RUN_NAME)

## Evaluate and Export

Dependency: `best.pt` or `last.pt` exists in Drive.

In [ ]:
best_pt = TRAIN_PROJECT / RUN_NAME / 'weights' / 'best.pt'
last_pt = TRAIN_PROJECT / RUN_NAME / 'weights' / 'last.pt'
weights = best_pt if best_pt.exists() else last_pt
assert weights.exists(), 'No checkpoint found. Run training first.'

split = 'test' if (DATA_YOLO_ROOT / 'images' / 'test').exists() else 'val'
model = YOLO(str(weights))
metrics = model.val(
    data=str(DATA_YAML),
    split=split,
    imgsz=IMGSZ,
    project=str(VAL_PROJECT),
    name=f'{RUN_NAME}_{split}',
    plots=True,
    exist_ok=True,
)

metrics_path = REPORTS_ROOT / f'{RUN_NAME}_{split}_metrics.json'
metrics_path.write_text(json.dumps(metrics.results_dict, indent=2), encoding='utf-8')
print('Metrics saved:', metrics_path)
print(metrics.results_dict)

try:
    exported = model.export(format='onnx', imgsz=IMGSZ)
    exported = Path(exported)
    target = EXPORTS_ROOT / exported.name
    if exported.exists() and exported.resolve() != target.resolve():
        shutil.copy2(exported, target)
    print('Exported model:', target if target.exists() else exported)
except Exception as exc:
    print('ONNX export skipped or failed:', exc)

## Final Backup Summary

Dependency: optional. This writes a small run summary into Drive.

In [ ]:
summary = {
    'drive_root': str(DRIVE_ROOT),
    'data_yaml': str(DATA_YAML),
    'train_dir': str(TRAIN_PROJECT / RUN_NAME),
    'best_pt': str(TRAIN_PROJECT / RUN_NAME / 'weights' / 'best.pt'),
    'last_pt': str(TRAIN_PROJECT / RUN_NAME / 'weights' / 'last.pt'),
}
summary_path = REPORTS_ROOT / f'{RUN_NAME}_run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(summary_path.read_text(encoding='utf-8'))